# Etapa 3 — Treinamento de modelos lineares e não lineares

Continuação do `02_selecao_atributos.ipynb`. Este notebook ajusta os modelos candidatos usando **somente o conjunto de treino** e os 23 atributos escolhidos na etapa 2. O conjunto de teste continua intocado: ele só entra na etapa 4 (avaliação e seleção de modelos).

Roteiro:
1. Configuração e dados
2. Modelos candidatos
3. Ajuste de hiperparâmetros por validação cruzada
4. Resultados

A lógica está em `src/train.py`.

## 1. Configuração e dados

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src import utils, preprocessing, features, train

utils.setup_plotting()

Os dados são reconstruídos com as mesmas funções da etapa 2 (nenhum dado processado é salvo em disco) e a divisão treino/teste é idêntica, porque usa a mesma semente.

In [ ]:
patients, drugs, data_dictionary = preprocessing.load_data()
variable_groups = preprocessing.get_variable_groups(patients)
target = preprocessing.create_target(patients)
target.index = patients["inpatient.number"].values

X, _ = features.build_model_matrix(patients, drugs, variable_groups)
X_train, X_test, y_train, y_test = preprocessing.split_train_test(X, target)

selected_features = features.load_selected_features()
X_train_selected = X_train[selected_features]

print(f"\nTreino: {X_train_selected.shape[0]} internações x {X_train_selected.shape[1]} atributos | eventos: {y_train.mean():.1%}")

## 2. Modelos candidatos

Dois modelos lineares e quatro não lineares, todos vistos na disciplina:

| Modelo | Tipo | Aula | Hiperparâmetros ajustados |
|---|---|---|---|
| Regressão logística com penalidade L2 (Ridge) | Linear | 04, 07 | `C` (inverso da penalidade) |
| Análise discriminante linear (LDA) | Linear | 04 | `shrinkage` (regularização da covariância) |
| SVM com kernel RBF | Não linear | 10 | `C`, `gamma` |
| Random Forest | Não linear | 09 | `max_features`, `min_samples_leaf` |
| Gradient Boosting | Não linear | 09 | nº de árvores, taxa de aprendizado, profundidade |
| K vizinhos mais próximos (KNN) | Não linear | 02 | `n_neighbors` (K), `weights` (peso uniforme ou pela distância) |

Todos são um `Pipeline` com **imputação pela mediana -> padronização -> classificador**. Assim a imputação e a padronização são reajustadas em cada *fold* de treino e nunca veem o *fold* de validação. A padronização é indispensável para logística, SVM e KNN (que se baseiam em distâncias ou no tamanho dos coeficientes); para as árvores é inócua.

## 3. Ajuste de hiperparâmetros por validação cruzada

**Objetivo**: para cada modelo, escolher a configuração com melhor desempenho esperado em dados novos.

**Como é feito**: busca em grade com validação cruzada estratificada de 10 *folds* (os mesmos *folds* para todos os modelos), métrica AUC. A melhor configuração de cada modelo é reajustada em todo o treino.

In [ ]:
cv_summary, searches = train.tune_models(X_train_selected, y_train)
cv_summary.drop(columns="best_params").to_csv(utils.RESULTS_DIR / "model_cv_results.csv")

## 4. Resultados

**Como ler a tabela**: uma linha por modelo, com a melhor configuração encontrada. `cv_auc_mean` é a AUC média nos 10 *folds* de validação e `cv_auc_se` o seu erro padrão; `train_auc_mean` é a AUC nos próprios dados de treino. Uma distância grande entre `train_auc_mean` e `cv_auc_mean` indica sobreajuste. Diferenças de `cv_auc_mean` menores que um erro padrão não distinguem dois modelos.

> **Atenção**: estas AUCs são **otimistas** e não servem para escolher o modelo. Os 23 atributos foram escolhidos usando todo o treino, inclusive as internações que aqui servem de validação, e por um modelo linear (o Lasso), o que favorece os modelos lineares na comparação. A comparação correta, com a seleção refeita dentro de cada *fold* (validação cruzada aninhada), está na seção 2 do notebook 04. O que este notebook entrega são os seis modelos ajustados, com seus hiperparâmetros.

In [ ]:
pd.set_option("display.max_colwidth", 80)
cv_summary

**AUC por *fold***. Como os *folds* são os mesmos para todos os modelos, a tabela permite comparar par a par: se um modelo ganha de outro em quase todos os *folds*, a diferença é consistente mesmo que as médias sejam próximas.

In [ ]:
fold_scores = train.cv_fold_scores(searches)
fold_scores.to_csv(utils.RESULTS_DIR / "model_cv_fold_scores.csv")
fold_scores.round(3)